# Paragraph typing: Random Forest training example

This notebook demonstrates the **manual** training workflow with a **synthetic example** for the Django keystroke project.
It uses `sample_data/raw/aggregated_paragraph.csv`, which contains extracted timing features, **not** individual raw key events. Django stores the key events and attempts in its private database.

The data, database, trained models, and identity mappings are private and intentionally absent from the public repository. First collect your own participants' data and export the aggregated paragraph CSV locally. Run Jupyter from the repository root or its `notebooks/` directory. See `requirements.txt` for runtime dependencies.

The historical experiment used `data/processed/processed_paragraph.csv`. This example starts from the aggregated file so that it can split sessions **before** fitting a scaler; historical accuracy and EER figures do not automatically apply to this revised experiment.


In [ ]:
from datetime import datetime, timezone
import json
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import sklearn
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score
from sklearn.preprocessing import LabelEncoder, StandardScaler

# Jupyter may launch in the repository root or in its notebooks/ directory.
PROJECT_ROOT = next(
    (candidate for candidate in (Path.cwd().resolve(), Path.cwd().resolve().parent)
     if (candidate / "manage.py").is_file()),
    None,
)
if PROJECT_ROOT is None:
    raise RuntimeError("Launch Jupyter from the project root or the notebooks/ directory.")

CSV_PATH = PROJECT_ROOT / "sample_data" / "raw" / "aggregated_paragraph.csv"
if not CSV_PATH.is_file():
    raise FileNotFoundError(
        "Missing sample_data/raw/aggregated_paragraph.csv. Copy the public synthetic example into the project."
    )
print("Input CSV found.")


## 1. Load features and split by collection session

The CSV has participant labels (`subject`), session information (`sessionIndex`, `rep`), and numeric timing features. Session 1 is used for training; session 2 is reserved for a small **identification** check. The source must contain at least one record for each participant in both sessions. With just one record per participant per session, the results have considerable uncertainty.


In [ ]:
df = pd.read_csv(CSV_PATH)
required = {"subject", "sessionIndex", "rep"}
missing = required - set(df.columns)
if missing:
    raise ValueError(f"Aggregated CSV is missing required columns: {sorted(missing)}")
if df["subject"].isna().any() or df["subject"].astype(str).str.strip().eq("").any():
    raise ValueError("Each training record requires a participant label.")

feature_columns = [
    column for column in df.columns
    if column not in {"subject", "participant", "subject_id", "sessionIndex", "rep"}
]
if not feature_columns:
    raise ValueError("No feature columns found in the aggregated CSV.")

# Match the collector's behavior: absent feature values become zero.
features = df[feature_columns].apply(pd.to_numeric, errors="coerce").fillna(0.0)
if not np.isfinite(features.to_numpy(dtype=float)).all():
    raise ValueError("Feature columns contain non-finite values.")

encoder = LabelEncoder().fit(df["subject"].astype(str))
labels = encoder.transform(df["subject"].astype(str))
sessions = pd.to_numeric(df["sessionIndex"], errors="raise")
train_mask = sessions.eq(1).to_numpy()
test_mask = sessions.eq(2).to_numpy()
if not train_mask.any() or not test_mask.any():
    raise ValueError("Collect both session 1 (training) and session 2 (evaluation).")
if set(labels[train_mask]) != set(labels[test_mask]):
    raise ValueError("Every participant must occur in both training and evaluation sessions.")

X_train_raw = features.loc[train_mask].to_numpy(dtype=float)
X_test_raw = features.loc[test_mask].to_numpy(dtype=float)
y_train = labels[train_mask]
y_test = labels[test_mask]
print(f"Participants: {len(encoder.classes_)}; features: {len(feature_columns)}")
print(f"Training rows: {len(y_train)}; evaluation rows: {len(y_test)}")


## 2. Fit preprocessing on training rows only

Save the exact same scaler used to transform training features. `collector/views.py` loads an external `scaler.pkl` before passing a live attempt into `model.pkl`. A fitted scaler must therefore accompany the model for this example. Never fit it on evaluation records.


In [ ]:
scaler = StandardScaler().fit(X_train_raw)
X_train = scaler.transform(X_train_raw)
X_test = scaler.transform(X_test_raw)

rf = RandomForestClassifier(
    n_estimators=300,
    max_depth=None,
    random_state=42,
    class_weight="balanced",
    n_jobs=-1,
)
rf.fit(X_train, y_train)

predicted = rf.predict(X_test)
print("Identification accuracy on session 2:", round(accuracy_score(y_test, predicted), 4))
print("Identification macro F1 on session 2:", round(f1_score(y_test, predicted, average="macro", zero_division=0), 4))
print("Correct/total:", int((predicted == y_test).sum()), "/", len(y_test))


## 3. Verification needs more independent data

`rf.predict_proba(X_test)` produces a class score for each claim. The application also needs a **per-person decision threshold** to accept or reject a claim. Do not estimate thresholds from session 2 and then report verification performance on the same session: that would reuse evaluation data for calibration. With only one session-2 sample per participant, the genuine score count is too small to validate an EER or a dependable threshold. Gather additional independent sessions for threshold calibration and final evaluation. No `thresholds.json` is generated here.

The existing application requires `model.pkl` and `thresholds.json` under `training/models/active_paragraph/`; the following cell intentionally writes **only a candidate**. Do not promote it directly into `active_paragraph` or mix it with an older threshold file.


In [ ]:
# Optional: show score matrix dimensions without disclosing participant identities.
scores = rf.predict_proba(X_test)
print("Evaluation score matrix (rows, enrolled classes):", scores.shape)


## 4. Save candidate artifacts locally

The candidate directory is excluded by the project's `/training/models/` ignore rule. `label_encoder.pkl` contains participant labels and must remain private. `meta.json` stores the feature order expected by `collector/views.py`. These files are useful for further local development and must be reviewed with independently calibrated thresholds before promotion.


In [ ]:
stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
candidate_dir = PROJECT_ROOT / "training" / "models" / "candidates" / f"rf_paragraph_{stamp}"
candidate_dir.mkdir(parents=True, exist_ok=False)

joblib.dump(rf, candidate_dir / "model.pkl")
joblib.dump(scaler, candidate_dir / "scaler.pkl")
joblib.dump(encoder, candidate_dir / "label_encoder.pkl")
meta = {
    "model_type": "RandomForestClassifier",
    "dataset": "paragraph",
    "training_session": 1,
    "evaluation_session": 2,
    "training_rows": int(len(y_train)),
    "evaluation_rows": int(len(y_test)),
    "n_features": len(feature_columns),
    "feature_cols": feature_columns,
    "score_type": "predict_proba",
    "sklearn_version": sklearn.__version__,
    "created_utc": stamp,
    "threshold_status": "not_calibrated",
}
(candidate_dir / "meta.json").write_text(json.dumps(meta, indent=2, ensure_ascii=False), encoding="utf-8")
print("Saved candidate artifacts to the private training/models/candidates/ directory.")
print("Verification thresholds have not been calibrated; no active model was replaced.")


## Share this notebook safely

Keep `data/`, `training/models/`, the database, and participant screenshots private. Before committing edited notebooks, clear their execution outputs and confirm that no personal names, absolute machine paths, credentials, or real measurements were copied into a cell. See the repository README for data collection and the local development workflow.

The bundled `sample_data/` is fictional. To work with consensually collected private data, change `CSV_PATH` in the first code cell to `PROJECT_ROOT / "data" / "raw" / "aggregated_paragraph.csv"`. Keep participant data and trained artifacts out of the public repository.
